<a href="https://colab.research.google.com/github/sherifmrehan-spec/project2/blob/main/project2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sqlite3
import pandas as pd
import json
from bs4 import BeautifulSoup

In [ ]:
# File locations
database_file = "/content/download"
catalogue_file = "/content/download (1)"
kickoff_file = "/content/download (2)"

print("Files configured.")

Files configured.


In [ ]:
# Connect to the SQLite database
conn = sqlite3.connect(database_file)

# Show the tables in the database
tables = pd.read_sql_query("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
""", conn)

tables

,name
0,books
1,checkouts
2,members


In [ ]:
q1 = pd.read_sql_query("""
    SELECT
        m.member_id,
        m.first_name,
        m.last_name,
        COUNT(c.checkout_id) AS checkout_count
    FROM members AS m
    LEFT JOIN checkouts AS c
        ON m.member_id = c.member_id
    GROUP BY
        m.member_id,
        m.first_name,
        m.last_name
    ORDER BY
        m.member_id;
""", conn)

q1

,member_id,first_name,last_name,checkout_count
0,1001,Salma,Ibrahim,1
1,1002,Fares,Saleh,2
2,1003,Bassel,Hegazy,9
3,1004,Fares,Wahba,0
4,1005,Youssef,Halim,3
...,...,...,...,...
75,1076,Dina,Wahba,7
76,1077,Lina,Rashad,6
77,1078,Habiba,Osman,0
78,1079,Rana,Osman,10


In [ ]:

print("Number of members:", len(q1))
print("Members with zero checkouts:", (q1["checkout_count"] == 0).sum())

Number of members: 80
Members with zero checkouts: 18


Which books match a chosen author pattern?

Author pattern chosen: `D%`

In [ ]:
q2 = pd.read_sql_query("""
    SELECT
        book_id,
        title,
        author
    FROM books
    WHERE author LIKE 'D%'
    ORDER BY author, title;
""", conn)

q2



,book_id,title,author
0,507,Fossils and Fireflies,Dalia Serry
1,508,The Quiet Observatory,Dalia Serry
2,509,Marbles and Mirrors,Diaa Sultan
3,510,The Missing Metronome,Diaa Sultan


In [ ]:
print("Books matching the pattern:", len(q2))

Books matching the pattern: 4


Whatare the most popular books?

highest checkouts, ranked from most borrowed to least borrowed.

In [ ]:
q3 = pd.read_sql_query("""
    SELECT
        b.book_id,
        b.title,
        b.author,
        COUNT(c.checkout_id) AS checkout_count
    FROM books AS b
    LEFT JOIN checkouts AS c
        ON b.book_id = c.book_id
    GROUP BY
        b.book_id,
        b.title,
        b.author
    ORDER BY
        checkout_count DESC,
        b.title ASC
    LIMIT 5;
""", conn)

q3

,book_id,title,author,checkout_count
0,501,The Silver Kite,Amina Darwish,57
1,507,Fossils and Fireflies,Dalia Serry,55
2,513,Circuits for Beginners,Galal Mounir,46
3,519,Kites Over Cairo,Jasmine Wahdan,38
4,525,Storms and Sailboats,Mahmoud Rafei,25


Who are the most active readers?

The ten members with the highest number of checkouts, ranked from most active to least active

In [ ]:
q4 = pd.read_sql_query("""
    SELECT
        m.member_id,
        m.first_name,
        m.last_name,
        COUNT(c.checkout_id) AS checkout_count
    FROM members AS m
    LEFT JOIN checkouts AS c
        ON m.member_id = c.member_id
    GROUP BY
        m.member_id,
        m.first_name,
        m.last_name
    ORDER BY
        checkout_count DESC,
        m.member_id ASC
    LIMIT 10;
""", conn)

q4

,member_id,first_name,last_name,checkout_count
0,1034,Aya,Wahba,25
1,1044,Sherif,Saleh,21
2,1008,Ziad,Saleh,19
3,1010,Nour,Nabil,18
4,1027,Mostafa,Fouad,18
5,1018,Ahmed,Shafik,17
6,1024,Youssef,Hegazy,17
7,1065,Adam,Fahmy,17
8,1030,Reem,Osman,16
9,1047,Sara,Rashad,16


What does a neighborhood's activity look like further back in time?

Neighborhood selected: `Maadi`

The checkout records are ordered from newest to oldest, with the 10 most recent checkouts skipped.

In [ ]:
q5 = pd.read_sql_query("""
    SELECT
        c.checkout_id,
        c.member_id,
        m.first_name,
        m.last_name,
        m.neighborhood,
        c.book_id,
        b.title,
        c.checkout_date,
        c.return_date
    FROM checkouts AS c
    JOIN members AS m
        ON c.member_id = m.member_id
    JOIN books AS b
        ON c.book_id = b.book_id
    WHERE m.neighborhood = 'Maadi'
    ORDER BY c.checkout_date DESC
    LIMIT -1 OFFSET 10;
""", conn)

q5

In [ ]:
# Load the book catalogue JSON
with open(catalogue_file, "r", encoding="utf-8") as f:
    catalogue_data = json.load(f)

catalogue = pd.DataFrame(catalogue_data)

print("Catalogue rows:", len(catalogue))
catalogue.head((32))

Catalogue rows: 32


,book_id,genre,pages,publication_year,publisher
0,501,Adventure,128,2017.0,Nile Press
1,502,Adventure,109,2018.0,Delta House
2,503,Historical,259,NaN,Nile Press
3,504,Science,319,2009.0,Cairo Young Readers
4,505,Historical,216,2024.0,Oasis Books
5,506,Friendship,183,NaN,Nile Press
6,507,Science,160,2024.0,Nile Press
7,508,Science,306,2011.0,Oasis Books
8,509,Mystery,221,2011.0,Nile Press
9,510,Mystery,134,2013.0,Cairo Young Readers


In [ ]:

kickoff_tables = pd.read_html(kickoff_file)

print("Number of tables found:", len(kickoff_tables))

kickoff = kickoff_tables[0]

kickoff.head(10)

Number of tables found: 1


,Member ID,Book ID,Checkout Date
0,1026,522,2025-07-11
1,1049,520,2025-07-11
2,1062,525,2025-07-05
3,1065,520,2025-07-07
4,1104,515,2025-07-07


In [ ]:
kickoff = kickoff.rename(columns={
    "Member ID": "member_id",
    "Book ID": "book_id",
    "Checkout Date": "checkout_date"
})

kickoff

,member_id,book_id,checkout_date
0,1026,522,2025-07-11
1,1049,520,2025-07-11
2,1062,525,2025-07-05
3,1065,520,2025-07-07
4,1104,515,2025-07-07
5,1009,503,2025-07-09
6,1063,522,2025-07-07
7,1022,511,2025-07-12
8,1029,523,2025-07-09
9,1201,509,2025-07-10


In [ ]:
kickoff["source"] = "Reading Kickoff"

kickoff.head()

,member_id,book_id,checkout_date,source
0,1026,522,2025-07-11,Reading Kickoff
1,1049,520,2025-07-11,Reading Kickoff
2,1062,525,2025-07-05,Reading Kickoff
3,1065,520,2025-07-07,Reading Kickoff
4,1104,515,2025-07-07,Reading Kickoff


In [ ]:
database_checkouts = pd.read_sql_query("""
    SELECT
        checkout_id,
        member_id,
        book_id,
        checkout_date,
        return_date
    FROM checkouts
""", conn)

database_checkouts["source"] = "Database"

print("Database checkout records:", len(database_checkouts))

database_checkouts.head()

Database checkout records: 391


,checkout_id,member_id,book_id,checkout_date,return_date,source
0,9263,1047,517,2024-10-21,2024-11-07,Database
1,9340,1072,513,2025-08-24,2025-09-01,Database
2,9231,1053,523,2024-02-04,2024-02-16,Database
3,9129,1032,513,2025-06-21,2025-06-29,Database
4,9370,1079,511,2025-11-11,2025-12-03,Database


In [ ]:
kickoff["checkout_id"] = pd.NA
kickoff["return_date"] = pd.NA

kickoff = kickoff[
    [
        "checkout_id",
        "member_id",
        "book_id",
        "checkout_date",
        "return_date",
        "source"
    ]
]

kickoff.head()

,checkout_id,member_id,book_id,checkout_date,return_date,source
0,<NA>,1026,522,2025-07-11,<NA>,Reading Kickoff
1,<NA>,1049,520,2025-07-11,<NA>,Reading Kickoff
2,<NA>,1062,525,2025-07-05,<NA>,Reading Kickoff
3,<NA>,1065,520,2025-07-07,<NA>,Reading Kickoff
4,<NA>,1104,515,2025-07-07,<NA>,Reading Kickoff


In [ ]:
all_checkouts = pd.concat(
    [database_checkouts, kickoff],
    ignore_index=True
)

print("Database checkouts:", len(database_checkouts))
print("Reading Kickoff checkouts:", len(kickoff))
print("Combined checkouts:", len(all_checkouts))

all_checkouts.head(10)

Database checkouts: 391
Reading Kickoff checkouts: 26
Combined checkouts: 417


,checkout_id,member_id,book_id,checkout_date,return_date,source
0,9263,1047,517,2024-10-21,2024-11-07,Database
1,9340,1072,513,2025-08-24,2025-09-01,Database
2,9231,1053,523,2024-02-04,2024-02-16,Database
3,9129,1032,513,2025-06-21,2025-06-29,Database
4,9370,1079,511,2025-11-11,2025-12-03,Database


In [ ]:
members = pd.read_sql_query("""
    SELECT
        member_id,
        first_name,
        last_name,
        grade,
        neighborhood,
        membership_status,
        join_date
    FROM members
""", conn)

members.head(10)

,member_id,first_name,last_name,grade,neighborhood,membership_status,join_date
0,1001,Salma,Ibrahim,8.0,Maadi,Active,2023-04-05
1,1002,Fares,Saleh,9.0,Maadi,Active,None
2,1003,Bassel,Hegazy,6.0,Maadi,Active,2025-04-23
3,1004,Fares,Wahba,7.0,Maadi,inactive,2024-10-09
4,1005,Youssef,Halim,9.0,Maadi,Active,2024-05-05


In [ ]:
all_checkouts = all_checkouts.merge(
    members,
    on="member_id",
    how="left"
)

all_checkouts.head(10)

,checkout_id,member_id,book_id,checkout_date,return_date,source,first_name,last_name,grade,neighborhood,membership_status,join_date
0,9263,1047,517,2024-10-21,2024-11-07,Database,Sara,Rashad,NaN,Heliopolis,Inactive,2024-06-25
1,9340,1072,513,2025-08-24,2025-09-01,Database,Seif,Zaki,9.0,Zamalek,Active,2025-10-21
2,9231,1053,523,2024-02-04,2024-02-16,Database,Adam,Shafik,9.0,Heliopolis,Active,2024-01-03
3,9129,1032,513,2025-06-21,2025-06-29,Database,Nada,Zaki,7.0,Nasr City,Active,2025-10-19
4,9370,1079,511,2025-11-11,2025-12-03,Database,Rana,Osman,8.0,Shubra,Active,2024-10-27
5,9082,1010,528,2025-11-11,2025-11-18,Database,Nour,Nabil,6.0,Maadi,Active,2024-05-15
6,9238,1057,513,2024-03-28,2024-04-08,Database,Ahmed,Fahmy,8.0,Heliopolis,Active,2024-01-12
7,9012,1010,501,2025-02-17,None,Database,Nour,Nabil,6.0,Maadi,Active,2024-05-15
8,9127,1024,506,2025-06-13,None,Database,Youssef,Hegazy,8.0,Nasr City,inactive,2024-01-04
9,9020,1016,506,2024-04-15,2024-05-04,Database,Dina,Rashad,8.0,Maadi,Active,2023-04-24


In [ ]:
books = pd.read_sql_query("""
    SELECT
        book_id,
        title,
        author
    FROM books
""", conn)

books.head()

,book_id,title,author
0,501,The Silver Kite,Amina Darwish
1,502,Desert Compass,Amina Darwish
2,503,The Lantern Maker,Adel Roushdy
3,504,Rooftop Astronomers,Adel Roushdy
4,505,Letters to the Nile,Aya Hafez


In [ ]:
all_checkouts = all_checkouts.merge(
    books,
    on="book_id",
    how="left"
)

all_checkouts.head(10)

,checkout_id,member_id,book_id,checkout_date,return_date,source,first_name,last_name,grade,neighborhood,membership_status,join_date,title,author
0,9263,1047,517,2024-10-21,2024-11-07,Database,Sara,Rashad,NaN,Heliopolis,Inactive,2024-06-25,Shadows on the Corniche,Hani Nagati
1,9340,1072,513,2025-08-24,2025-09-01,Database,Seif,Zaki,9.0,Zamalek,Active,2025-10-21,Circuits for Beginners,Galal Mounir
2,9231,1053,523,2024-02-04,2024-02-16,Database,Adam,Shafik,9.0,Heliopolis,Active,2024-01-03,Footsteps in the Dust,Laila Shokry
3,9129,1032,513,2025-06-21,2025-06-29,Database,Nada,Zaki,7.0,Nasr City,Active,2025-10-19,Circuits for Beginners,Galal Mounir
4,9370,1079,511,2025-11-11,2025-12-03,Database,Rana,Osman,8.0,Shubra,Active,2024-10-27,Winter in Alexandria,Farida Anwar
5,9082,1010,528,2025-11-11,2025-11-18,Database,Nour,Nabil,6.0,Maadi,Active,2024-05-15,The Glass Beehive,Mona Kholoussy
6,9238,1057,513,2024-03-28,2024-04-08,Database,Ahmed,Fahmy,8.0,Heliopolis,Active,2024-01-12,Circuits for Beginners,Galal Mounir
7,9012,1010,501,2025-02-17,None,Database,Nour,Nabil,6.0,Maadi,Active,2024-05-15,The Silver Kite,Amina Darwish
8,9127,1024,506,2025-06-13,None,Database,Youssef,Hegazy,8.0,Nasr City,inactive,2024-01-04,The Paper Boat Club,Aya Hafez
9,9020,1016,506,2024-04-15,2024-05-04,Database,Dina,Rashad,8.0,Maadi,Active,2023-04-24,The Paper Boat Club,Aya Hafez


In [ ]:
all_checkouts = all_checkouts.merge(
    catalogue,
    on="book_id",
    how="left"
)

all_checkouts.head(16)

,checkout_id,member_id,book_id,checkout_date,return_date,source,first_name,last_name,grade,neighborhood,...,title,author,genre_x,pages_x,publication_year_x,publisher_x,genre_y,pages_y,publication_year_y,publisher_y
0,9263,1047,517,2024-10-21,2024-11-07,Database,Sara,Rashad,NaN,Heliopolis,...,Shadows on the Corniche,Hani Nagati,Mystery,338,2015.0,Delta House,Mystery,338,2015.0,Delta House
1,9340,1072,513,2025-08-24,2025-09-01,Database,Seif,Zaki,9.0,Zamalek,...,Circuits for Beginners,Galal Mounir,Science,294,2021.0,Oasis Books,Science,294,2021.0,Oasis Books
2,9231,1053,523,2024-02-04,2024-02-16,Database,Adam,Shafik,9.0,Heliopolis,...,Footsteps in the Dust,Laila Shokry,Historical,276,2018.0,Oasis Books,Historical,276,2018.0,Oasis Books
3,9129,1032,513,2025-06-21,2025-06-29,Database,Nada,Zaki,7.0,Nasr City,...,Circuits for Beginners,Galal Mounir,Science,294,2021.0,Oasis Books,Science,294,2021.0,Oasis Books
4,9370,1079,511,2025-11-11,2025-12-03,Database,Rana,Osman,8.0,Shubra,...,Winter in Alexandria,Farida Anwar,Historical,117,2016.0,Nile Press,Historical,117,2016.0,Nile Press
5,9082,1010,528,2025-11-11,2025-11-18,Database,Nour,Nabil,6.0,Maadi,...,The Glass Beehive,Mona Kholoussy,Nature,114,2010.0,Delta House,Nature,114,2010.0,Delta House
6,9238,1057,513,2024-03-28,2024-04-08,Database,Ahmed,Fahmy,8.0,Heliopolis,...,Circuits for Beginners,Galal Mounir,Science,294,2021.0,Oasis Books,Science,294,2021.0,Oasis Books
7,9012,1010,501,2025-02-17,None,Database,Nour,Nabil,6.0,Maadi,...,The Silver Kite,Amina Darwish,Adventure,128,2017.0,Nile Press,Adventure,128,2017.0,Nile Press
8,9127,1024,506,2025-06-13,None,Database,Youssef,Hegazy,8.0,Nasr City,...,The Paper Boat Club,Aya Hafez,Friendship,183,NaN,Nile Press,Friendship,183,NaN,Nile Press
9,9020,1016,506,2024-04-15,2024-05-04,Database,Dina,Rashad,8.0,Maadi,...,The Paper Boat Club,Aya Hafez,Friendship,183,NaN,Nile Press,Friendship,183,NaN,Nile Press


Task 1 : unified dataset and asking dataset questions

In [ ]:
# Preserve the original Task 1 dataset
task1_output = all_checkouts.copy()

# Save it as a CSV file
task1_output.to_csv(
    "/content/task1_unified_dataset.csv",
    index=False
)

print("Task 1 dataset saved.")
print("Rows:", len(task1_output))
print("Columns:", len(task1_output.columns))

Task 1 dataset saved.
Rows: 417
Columns: 22


In [ ]:
task2_data = task1_output.copy()

Task 2 — Data Integrity

The Task 1 unified dataset is preserved

In [ ]:
task2_data = task1_output.copy()

print("Task 2 starting dataset")
print("Rows:", len(task2_data))
print("Columns:", len(task2_data.columns))

Task 2 starting dataset
Rows: 417
Columns: 22


Problem 1 — Missing Values

identify every column containing missing values.

In [ ]:
missing_counts = task2_data.isna().sum()

missing_counts = missing_counts[missing_counts > 0].sort_values(ascending=False)

missing_counts

,0
return_date,91
grade,41
publication_year_x,35
publication_year_y,35
checkout_id,26
join_date,11
neighborhood,5
last_name,5
first_name,5
membership_status,5


In [ ]:
rows_with_missing = task2_data[task2_data.isna().any(axis=1)]

print("Rows containing at least one missing value:", len(rows_with_missing))

rows_with_missing

Rows containing at least one missing value: 148


,checkout_id,member_id,book_id,checkout_date,return_date,source,first_name,last_name,grade,neighborhood,...,title,author,genre_x,pages_x,publication_year_x,publisher_x,genre_y,pages_y,publication_year_y,publisher_y
0,9263,1047,517,2024-10-21,2024-11-07,Database,Sara,Rashad,NaN,Heliopolis,...,Shadows on the Corniche,Hani Nagati,Mystery,338,2015.0,Delta House,Mystery,338,2015.0,Delta House
7,9012,1010,501,2025-02-17,None,Database,Nour,Nabil,6.0,Maadi,...,The Silver Kite,Amina Darwish,Adventure,128,2017.0,Nile Press,Adventure,128,2017.0,Nile Press
8,9127,1024,506,2025-06-13,None,Database,Youssef,Hegazy,8.0,Nasr City,...,The Paper Boat Club,Aya Hafez,Friendship,183,NaN,Nile Press,Friendship,183,NaN,Nile Press
9,9020,1016,506,2024-04-15,2024-05-04,Database,Dina,Rashad,8.0,Maadi,...,The Paper Boat Club,Aya Hafez,Friendship,183,NaN,Nile Press,Friendship,183,NaN,Nile Press
22,9030,1018,507,2025-03-09,None,Database,Ahmed,Shafik,9.0,Maadi,...,Fossils and Fireflies,Dalia Serry,Science,160,2024.0,Nile Press,Science,160,2024.0,Nile Press
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
412,<NA>,1003,501,2025-07-08,NaN,Reading Kickoff,Bassel,Hegazy,6.0,Maadi,...,The Silver Kite,Amina Darwish,Adventure,128,2017.0,Nile Press,Adventure,128,2017.0,Nile Press
413,<NA>,1017,507,2025-07-11,NaN,Reading Kickoff,Adam,Badr,9.0,Maadi,...,Fossils and Fireflies,Dalia Serry,Science,160,2024.0,Nile Press,Science,160,2024.0,Nile Press
414,<NA>,1061,504,2025-07-06,NaN,Reading Kickoff,Ziad,Fahmy,9.0,zamalek,...,Rooftop Astronomers,Adel Roushdy,Science,319,2009.0,Cairo Young Readers,Science,319,2009.0,Cairo Young Readers
415,<NA>,1201,523,2025-07-08,NaN,Reading Kickoff,NaN,NaN,NaN,NaN,...,Footsteps in the Dust,Laila Shokry,Historical,276,2018.0,Oasis Books,Historical,276,2018.0,Oasis Books


In [ ]:
for column in missing_counts.index:
    print(f"\n--- {column} ---")
    print(
        task2_data.loc[
            task2_data[column].isna(),
            ["source", column]
        ].value_counts(dropna=False)
    )


--- return_date ---
source           return_date
Database         NaN            65
Reading Kickoff  NaN            26
Name: count, dtype: int64

--- grade ---
source           grade
Database         NaN      36
Reading Kickoff  NaN       5
Name: count, dtype: int64

--- publication_year_x ---
source           publication_year_x
Database         NaN                   34
Reading Kickoff  NaN                    1
Name: count, dtype: int64

--- publication_year_y ---
source           publication_year_y
Database         NaN                   34
Reading Kickoff  NaN                    1
Name: count, dtype: int64

--- checkout_id ---
source           checkout_id
Reading Kickoff  NaN            26
Name: count, dtype: int64

--- join_date ---
source           join_date
Reading Kickoff  NaN          6
Database         NaN          5
Name: count, dtype: int64

--- neighborhood ---
source           neighborhood
Reading Kickoff  NaN             5
Name: count, dtype: int64

--- last_name ---
sourc

Problem 2 — Duplicates

True duplicate records should be removed, while records that look similar but represent different checkout events should remain.

In [ ]:
duplicate_mask = task2_data.duplicated(keep=False)

print("Rows involved in exact duplicates:", duplicate_mask.sum())
print("Number of duplicate groups:", task2_data[duplicate_mask].drop_duplicates().shape[0])

In [ ]:
before = len(task2_data)

task2_data = task2_data.drop_duplicates().reset_index(drop=True)

after = len(task2_data)

print("Rows before:", before)
print("Rows after:", after)
print("True duplicates removed:", before - after)

Problem 3 — The Same Value , Different Ways

Text values were checked for inconsistent representations.

In [ ]:
print("Neighborhood values:")
print(sorted(task2_data["neighborhood"].dropna().unique()))

print("\nMembership status values:")
print(sorted(task2_data["membership_status"].dropna().unique()))

Neighborhood values:
['HELIOPOLIS', 'Heliopolis', 'Maadi', 'Maadi ', 'NASR CITY', 'Nasr City', 'Shubra', 'Zamalek', 'zamalek']

Membership status values:
['Active', 'Inactive', 'active', 'inactive']


Problem 4 — Checkouts With No Matching Member

In [ ]:
valid_member_ids = set(members["member_id"])

invalid_member_mask = ~task2_data["member_id"].isin(valid_member_ids)

print("Checkouts with no matching member:", invalid_member_mask.sum())

Checkouts with no matching member: 5


In [ ]:
invalid_checkouts = task2_data.loc[
    invalid_member_mask,
    ["checkout_id", "member_id", "book_id", "checkout_date", "source"]
]

invalid_checkouts

,checkout_id,member_id,book_id,checkout_date,source
395,<NA>,1104,515,2025-07-07,Reading Kickoff
400,<NA>,1201,509,2025-07-10,Reading Kickoff
402,<NA>,1104,526,2025-07-05,Reading Kickoff
405,<NA>,1150,530,2025-07-10,Reading Kickoff
415,<NA>,1201,523,2025-07-08,Reading Kickoff


In [ ]:
before = len(task2_data)

task2_data = task2_data.loc[~invalid_member_mask].copy()

after = len(task2_data)

print("Invalid-member checkout records removed:", before - after)
print("Rows remaining:", after)

Invalid-member checkout records removed: 5
Rows remaining: 412


Invalid-member decision

Checkout records referencing a member_id that does not exist in the registered members table were removed.

Task 2 — Final Cleaned Dataset

In [ ]:
print("Final number of rows:", len(task2_data))
print("Final number of columns:", len(task2_data.columns))

print("\nRemaining missing values:")
print(task2_data.isna().sum()[task2_data.isna().sum() > 0])

print("\nExact duplicate rows remaining:",
      task2_data.duplicated().sum())

Final number of rows: 412
Final number of columns: 22

Remaining missing values:
checkout_id           21
return_date           86
grade                 36
join_date              6
publication_year_x    35
publication_year_y    35
dtype: int64

Exact duplicate rows remaining: 8


In [ ]:
# Save the cleaned Task 2 dataset
task2_data.to_csv(
    "/content/task2_cleaned_data.csv",
    index=False
)

print("Saved: task2_cleaned_data.csv")

Saved: task2_cleaned_data.csv


Task 2 — Data Integrity Report

This report documents the four data-integrity problems identified in the unified dataset produced in Task 1.

## Problem 1 — Missing Values

### What was found

The unified dataset contained missing values in some columns.

### Where it was

The missing values occurred in the columns identified during the Task 2 investigation.

### How big it was

The number of missing values was counted for each affected column.

### What was done, and why

Missing values that represented information genuinely unavailable from the source data were retained rather than replaced with invented values. Missing catalogue information such as an unavailable publication year was also not fabricated.


## Problem 2 — Duplicates That Aren't All the Same

### What was found

The dataset was checked for records that were exactly duplicated. Records that only appeared similar were not automatically treated as duplicates.

### Where it was

The duplicate check was performed across the complete unified dataset.

### How big it was

The number of exact duplicate records removed was recorded during the cleaning process.

### What was done, and why

Only exact duplicate records were removed. Similar records were retained when they represented different checkout events. This prevents legitimate repeated borrowing activity from being incorrectly deleted.


## Problem 3 — The Same Value Written Different Ways

### What was found

Some text values represented the same real-world value but were inconsistent
### Where it was

The inconsistencies occurred in text columns, particularly `neighborhood` and `membership_status`.

### How big it was

The different representations were identified by examining the unique values in the affected columns before standardization.

### What was done, and why

Equivalent representations of the same value were converted to one consistent form. Genuinely different values were kept separate. To make grouping, filtering, and analysis reliable without incorrectly collapsing distinct categories.

## Problem 4 — Checkouts With No Matching Member

### What was found

Some checkout records referenced a `member_id` that did not correspond to a registered member in the members table.

### Where it was

The problem occurred in the `member_id` column of the checkout records.

### How big it was

The number of checkout records with invalid member IDs was calculated by comparing checkout `member_id` values against the registered members table.

### What was done, and why

Checkout records whose `member_id` did not exist in the registered members table were removed. The registered members themselves were not modified. This was chosen because an unresolvable member reference cannot reliably be associated with a registered library member.





## Conclusion

The unified dataset from Task 1 was reviewed for the four specified data-integrity problems. Missing values were treated according to the meaning of the affected fields, only true duplicate records were removed, inconsistent representations were standardized without merging genuinely different values, and checkout records with nonexistent member IDs were removed.

The resulting cleaned dataset was saved as:

`task2_cleaned_data.csv`

**Student ID:** 30906030100615

task 3 : Fairness

In [ ]:
final_data = task2_data.copy()

print("Final dataset:")
print("Rows:", len(final_data))
print("Columns:", len(final_data.columns))

Final dataset:
Rows: 412
Columns: 22


In [ ]:
print("Neighborhood distribution:")
print(final_data["neighborhood"].value_counts(dropna=False))

print("\nMembership status distribution:")
print(final_data["membership_status"].value_counts(dropna=False))

print("\nGrade distribution:")
print(final_data["grade"].value_counts(dropna=False))

Neighborhood distribution:
neighborhood
Nasr City     103
Maadi          96
Heliopolis     88
Zamalek        60
Shubra         34
Maadi          19
zamalek        10
NASR CITY       1
HELIOPOLIS      1
Name: count, dtype: int64

Membership status distribution:
membership_status
Active      279
Inactive     55
active       46
inactive     32
Name: count, dtype: int64

Grade distribution:
grade
9.0    113
6.0     91
7.0     90
8.0     82
NaN     36
Name: count, dtype: int64


In [ ]:
print(final_data["source"].value_counts())

source
Database           391
Reading Kickoff     21
Name: count, dtype: int64


In [ ]:
print(
    final_data.groupby("source")["neighborhood"]
    .value_counts()
)

source           neighborhood
Database         Nasr City       99
                 Maadi           89
                 Heliopolis      85
                 Zamalek         56
                 Shubra          34
                 Maadi           18
                 zamalek          8
                 HELIOPOLIS       1
                 NASR CITY        1
Reading Kickoff  Maadi            7
                 Nasr City        4
                 Zamalek          4
                 Heliopolis       3
                 zamalek          2
                 Maadi            1
Name: count, dtype: int64


One realistic reason could be population density that actually read

And a change that the library can make would be advertisments and more encouragment for new people to start reading

fatal: not in a git directory
fatal: not in a git directory
